# neo4j-nano demo

DataFrames in, Cypher out. Neo4j Virtual Graphs embedded directly in Python.

In [ ]:
%pip install pandas jpype1

## Load data from CSV

In [ ]:
import pandas as pd
movies_df = pd.read_csv("data/movies.csv")
people_df = pd.read_csv("data/people.csv")
acted_in_df = pd.read_csv("data/acted_in.csv")
directed_df = pd.read_csv("data/directed.csv")


## Build graph and start engine

In [ ]:
from neo4j_nano import GraphEngine
engine = GraphEngine(accept_license=True)

# Nodes
engine.add_nodes(movies_df, label="Movie", id_column="movie_id")
engine.add_nodes(people_df, label="Person", id_column="person_id")

# Relationships
engine.add_relationships(acted_in_df, type="ACTED_IN",
    source_column="person_id", source_label="Person",
    target_column="movie_id", target_label="Movie")
engine.add_relationships(directed_df, type="DIRECTED",
    source_column="person_id", source_label="Person",
    target_column="movie_id", target_label="Movie")

engine.start()

## Query: Movies after 2000

In [ ]:
pd.DataFrame(engine.query("""
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    WHERE m.release_year > 2000
    RETURN p.name AS actor, m.title AS movie, m.release_year AS year
"""))

## Query: All paths

In [ ]:
pd.DataFrame(engine.query("""
    MATCH (p:Person)-[r]->(m:Movie)
    RETURN p.name AS person, type(r) AS relationship, m.title AS movie
    ORDER BY person, movie
"""))

## Query: Co-actors

In [ ]:
pd.DataFrame(engine.query("""
    MATCH (p1:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(p2:Person)
    WHERE p1.name < p2.name
    RETURN p1.name AS actor1, p2.name AS actor2, m.title AS movie
"""))

## Cleanup

In [ ]:
engine.stop()